# Items and reliability

Reproduces steps 2–3 of the Scale Validation Lab for the simulated *Academic Writing with AI* questionnaire, before and after fixing the reverse-worded item.

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "../analysis")
import psychometrics as ps

DATA = Path("../web/public/data")
pd.set_option("display.precision", 3)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

def load(name):
    df = pd.read_csv(DATA / f"{name}.csv")
    meta = json.loads((DATA / f"{name}.json").read_text())
    return df, meta


In [2]:
df, meta = load("writing_ai_simulated")
print(meta["name"], "| n =", len(df), "| complete cases:", len(df.dropna()))
desc = df.agg(["mean", "std", "skew"]).T
desc["missing_%"] = 100 * df.isna().mean()
desc

Academic Writing with AI questionnaire (simulated) | n = 320 | complete cases: 243


,mean,std,skew,missing_%
SE1,3.283,1.100,-0.275,1.562
SE2,3.225,1.145,-0.181,1.250
SE3,3.213,1.075,-0.138,1.562
SE4,3.211,1.104,-0.238,2.188
SE5,3.199,1.119,-0.149,2.500
CE1,3.331,1.117,-0.221,2.812
CE2,3.256,1.087,-0.166,1.250
CE3,3.270,1.083,-0.063,2.812
CE4,3.217,1.111,-0.156,2.188
CE5,3.212,1.052,0.094,1.250


In [3]:
subscales = {}
for it in meta["items"]:
    subscales.setdefault(it["subscale"], []).append(df.columns.get_loc(it["id"]))

def reliability(X_all, reverse_cols=()):
    rows = []
    X_all = ps.reverse(X_all, list(reverse_cols), *meta["scale"])
    for name, cols in subscales.items():
        X = ps.listwise(X_all[:, cols])
        a = ps.cronbach_alpha(X)
        lo, hi = ps.alpha_ci(a, len(X), len(cols))
        rows.append({"subscale": name, "n": len(X), "alpha": a, "ci_low": lo, "ci_high": hi, "omega": ps.omega_total(ps.corr(X))})
    return pd.DataFrame(rows)

X = df.to_numpy(float)
print("As supplied (CE4 not reverse-scored):")
display(reliability(X))
print("After reverse-scoring CE4:")
display(reliability(X, [df.columns.get_loc("CE4")]))

As supplied (CE4 not reverse-scored):


,subscale,n,alpha,ci_low,ci_high,omega
0,Writing self-efficacy,292,0.766,0.721,0.806,0.777
1,Critical evaluation of AI output,290,0.319,0.187,0.436,0.593
2,Dependence on AI,291,0.832,0.800,0.861,0.834


After reverse-scoring CE4:


,subscale,n,alpha,ci_low,ci_high,omega
0,Writing self-efficacy,292,0.766,0.721,0.806,0.777
1,Critical evaluation of AI output,290,0.798,0.759,0.833,0.799
2,Dependence on AI,291,0.832,0.800,0.861,0.834


In [4]:
cols = subscales["Critical evaluation of AI output"]
Xc = ps.listwise(X[:, cols])
pd.DataFrame(ps.item_stats(Xc), index=df.columns[cols]).rename(columns={"citc": "corrected item-total r", "alphaIfDeleted": "alpha if deleted"})

,corrected item-total r,alpha if deleted
CE1,0.486,-0.073
CE2,0.457,-0.032
CE3,0.412,0.019
CE4,-0.573,0.762
CE5,0.407,0.037
